# EV-Side MaskablePPO Grid Advisory Runbook

This notebook is the safe local runbook for the EV-side RL setup. It does not train by default. The agent sees one charging request, sees all Dundee stations as actions, uses an action mask so impossible stations cannot be chosen, and receives a reward based on user-side quality plus optional grid advisory feedback from DigitalTwin.

## Plain-Language Model

- Environment: the simulator state for one request at a time.
- Agent: the policy that chooses one station.
- Action: station index in the fixed Dundee station list.
- Action mask: true only for stations that are feasible for the request.
- Reward: positive for serving the request, reduced by cost, distance, wait, duration, low headroom, and grid risk.
- Grid advisory: disabled, recorded replay, or local HTTP API from DigitalTwin.

In [ ]:
from pathlib import Path
import importlib.util
import json
import random
import subprocess
import sys
from urllib import request as urlrequest, error as urlerror

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
EV_CORE_SRC = REPO_ROOT / 'packages' / 'ev_core' / 'src'
if str(EV_CORE_SRC) not in sys.path:
    sys.path.insert(0, str(EV_CORE_SRC))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RUN_TRAINING = False
GRID_ADVISORY_BASE_URL = 'http://127.0.0.1:8091'
print(REPO_ROOT)

In [ ]:
for package_name in ['gymnasium', 'stable_baselines3', 'sb3_contrib', 'tensorboard', 'pandas', 'pyarrow']:
    print(f'{package_name}: {bool(importlib.util.find_spec(package_name))}')

In [ ]:
from ev_core.data.repositories import DundeeSimulationRepository

repository = DundeeSimulationRepository(REPO_ROOT)
bundle = repository.load_bundle()
print({
    'stations': len(bundle.stations),
    'chargepoints': len(bundle.chargepoints),
    'requests_2023': len(bundle.replay_requests_2023),
    'requests_2024': len(bundle.replay_requests_2024),
    'transformers': len(bundle.transformers),
})

In [ ]:
if importlib.util.find_spec('gymnasium') is None:
    print('Gymnasium is not installed in this interpreter. Install requirements before env smoke checks or training.')
else:
    from ev_core.rl.env import DundeeStationSelectionEnv
    from ev_core.rl.scenarios import RLScenarioSampler

    sampler = RLScenarioSampler(bundle=bundle)
    scenario = sampler.sample(seed=1000, split='train', duration_hours=1, demand_level='normal')
    env = DundeeStationSelectionEnv(
        repo_root=REPO_ROOT,
        scenario=scenario,
        bundle=bundle,
        grid_advisory_mode='disabled',
    )
    observation, info = env.reset(seed=scenario.seed)
    mask = env.action_masks()
    print('scenario_id:', scenario.scenario_id)
    print('observation_shape:', observation.shape)
    print('observation_space_shape:', env.observation_space.shape)
    print('valid_action_count:', sum(mask))
    print('first_20_mask_values:', mask[:20])
    print('info:', {key: info[key] for key in ['scenario_id', 'request_index', 'valid_action_count', 'grid_advisory_mode']})

In [ ]:
if 'env' not in globals():
    print('Skipping rollout because env was not created.')
else:
    rng = random.Random(1000)
    total_reward = 0.0
    terminated = False
    steps = 0
    while not terminated and steps < 5:
        mask = env.action_masks()
        valid_actions = [index for index, allowed in enumerate(mask) if allowed]
        action = valid_actions[rng.randrange(len(valid_actions))] if valid_actions else 0
        observation, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        steps += 1
        print({'step': steps, 'action': action, 'reward': reward, 'selected_station_id': info['selected_station_id']})
    print({'steps': steps, 'total_reward': round(total_reward, 6), 'terminated': terminated})

In [ ]:
def http_json(method, url, payload=None):
    data = None if payload is None else json.dumps(payload).encode('utf-8')
    headers = {'Content-Type': 'application/json', 'Accept': 'application/json'}
    req = urlrequest.Request(url, data=data, headers=headers, method=method)
    with urlrequest.urlopen(req, timeout=2.0) as response:
        return json.loads(response.read().decode('utf-8'))

try:
    print(http_json('GET', f'{GRID_ADVISORY_BASE_URL}/v1/health'))
    proposal = {
        'request_id': 'notebook_request_001',
        'episode_id': 'notebook_smoke',
        'station_id': 'dundee_station_demo',
        'area_id': 'dundee_area_demo',
        'start_timestamp': '2024-03-15T18:00:00Z',
        'timebase_minutes': 30,
        'duration_steps': 2,
        'requested_energy_kwh': 24.0,
        'charger_kw': 22.0,
        'ev_schedule': [{'time_index': 0, 'p_kw': 22.0, 'q_kvar': 0.0}],
    }
    print(http_json('POST', f'{GRID_ADVISORY_BASE_URL}/v1/proposals/evaluate', proposal))
except Exception as exc:
    print('Grid advisory API is not running yet.')
    print('Start it from DigitalTwin root with: python scripts/serve_grid_advisory_api.py --port 8091')
    print(type(exc).__name__, exc)

In [ ]:
training_command = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'rl_training' / 'train_maskable_ppo_station_selector.py'),
    '--repo-root', str(REPO_ROOT),
    '--output-dir', str(REPO_ROOT / 'models' / 'rl'),
    '--total-timesteps', '10000',
    '--seed-start', '1000',
    '--scenario-count', '4',
    '--grid-advisory-mode', 'disabled',
]
print(' '.join(training_command))

In [ ]:
if RUN_TRAINING:
    subprocess.run(training_command, check=True, cwd=REPO_ROOT)
else:
    print('RUN_TRAINING is False, so no model training was started.')